# TP 4 — Spark SQL : requêter le fil rouge e-commerce

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

## Consignes
- Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (remplacez les `...`).
- Rédigez vos réponses dans les cellules *Votre réponse :*.
- Le notebook doit s'exécuter **de bout en bout** (Kernel > Restart & Run All) avant d'être poussé.
- Livrable : `notebooks/TP4_spark_sql.ipynb` **avec les sorties visibles**, poussé sur votre dépôt avant la séance 5.

## Déroulé
| Partie | Contenu | Durée |
|---|---|---|
| A | Mise en place : données, SparkSession, vues | 15 min |
| B | Premières requêtes SQL | 25 min |
| C | L'enquête FCFA | 30 min |
| D | Les indicateurs de la direction | 40 min |
| E | SQL ou API ? `explain()` tranche | 20 min |
| F | Discussion et quiz | 20 min |

## 0. Vérification de l'environnement

Données : si le dossier `data/` est absent, exécutez d'abord dans un terminal (ou une cellule `!`) :
```
python generate_data.py --scale 0.1 --outdir data
```
Graine 42 : tous les étudiants ont **exactement** les mêmes données.

In [43]:
import sys
print("Python :", sys.version.split()[0])

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("TP4-SparkSQL")
         .master("local[*]")
         .getOrCreate())
print("Spark  :", spark.version)
spark.sparkContext.setLogLevel("ERROR")

Python : 3.12.10
Spark  : 4.2.0


### Tableau de relevés

Il se remplit **au fil du TP** ; la dernière cellule du notebook l'affiche. Un notebook sans chiffres n'est pas un livrable.

In [44]:
releves = {
    "A_nb_lignes_clients":        None,
    "A_nb_lignes_commandes":      None,
    "A_type_montant_total_fcfa":  None,   # ex. "string"
    "C2_nb_valeurs_polluees":     None,
    "C1_ca_naif":                 None,
    "C4_ca_nettoye":              None,
    "C5_ecart_fcfa":              None,
    "C5_ecart_pct":               None,
    "D1_part_ca_livree_pct":      None,
    "D2_mois_record":             None,
    "D3_panier_moyen_mobile":     None,
    "D5_part_mobile_money_pct":   None,
}

## Partie A — Mise en place (15 min)

### A.1 — Charger les quatre sources et créer les vues

Chargez `customers.csv`, `orders.csv`, `products.csv` (CSV : `header=True`, `inferSchema=True`) et `payments.json`, puis créez les vues temporaires `clients`, `commandes`, `produits`, `paiements`.

In [45]:
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.ansi.enabled", "false")

In [46]:
base = "../data/"

clients   = spark.read.csv(base + "customers.csv", header=True, inferSchema=True)
commandes = spark.read.csv(base + "orders.csv", header=True, inferSchema=True)
produits  = spark.read.csv(base + "products.csv", header=True, inferSchema=True)
paiements = spark.read.json(base + "payments.json")

clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")
produits.createOrReplaceTempView("produits")
paiements.createOrReplaceTempView("paiements")

spark.catalog.listTables()

[Table(name='clients', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='commandes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='commandes_clean', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='paiements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='produits', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### A.2 — Premier relevé

Comptez les lignes de `clients` et `commandes`, affichez le schéma de `commandes`, et relevez le **type inféré** de `montant_total_fcfa` et de `frais_livraison_fcfa`.

In [47]:
# === À COMPLÉTER ===
releves["A_nb_lignes_clients"]   = spark.sql("SELECT COUNT(*) AS n FROM clients").first()["n"]
releves["A_nb_lignes_commandes"] = ...

commandes.printSchema()
releves["A_type_montant_total_fcfa"] = ...   # recopiez le type lu dans le schema
print(releves)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

{'A_nb_lignes_clients': 5025, 'A_nb_lignes_commandes': Ellipsis, 'A_type_montant_total_fcfa': Ellipsis, 'C2_nb_valeurs_polluees': None, 'C1_ca_naif': None, 'C4_ca_nettoye': None, 'C5_ecart_fcfa': None, 'C5_ecart_pct': None, 'D1_part_ca_livree_pct': None, 'D2_mois_record': None, 'D3_panier_moyen_mobile': None, 'D5_part_mobile_money_pct': None}


**Question A** — Une des deux colonnes de montants n'a pas le type attendu. Laquelle, et qu'en déduisez-vous sur le contenu du fichier ? (Vous vérifierez votre hypothèse en partie C.)

*Votre réponse :*

…

## Partie B — Premières requêtes SQL (25 min)

Une requête par cellule, résultat affiché avec `.show()`.

### B1 — Les 10 premiers clients de Dakar
Colonnes : `customer_id`, `prenom`, `nom`, `ville`.

In [48]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT customer_id, prenom, nom, ville
    FROM clients
    WHERE ville = 'Dakar'
    LIMIT 10
""").show()

+-----------+--------+--------+-----+
|customer_id|  prenom|     nom|ville|
+-----------+--------+--------+-----+
|    C000878|  Yacine|    Faye|Dakar|
|    C002485|   Astou|  Ndiaye|Dakar|
|    C000812|  Diarra|    Wade|Dakar|
|    C000249|Seynabou|Goudiaby|Dakar|
|    C003299|   Adama|    Fall|Dakar|
|    C001282|  Yacine|      Sy|Dakar|
|    C000334|Maguette|   Mendy|Dakar|
|    C002548| Rokhaya|   Badji|Dakar|
|    C002842|    Omar|  Diallo|Dakar|
|    C001159|  Sokhna|   Dieng|Dakar|
+-----------+--------+--------+-----+



### B2 — Combien de villes distinctes dans `clients` ?

In [49]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT COUNT(DISTINCT ville) AS nb_villes_distinctes
    FROM clients
""").show()

+--------------------+
|nb_villes_distinctes|
+--------------------+
|                  56|
+--------------------+



### B3 — Top 10 des produits les plus chers
Nom et prix, tri décroissant.

In [50]:
spark.sql("""
    SELECT nom_produit, prix_unitaire_fcfa
    FROM produits
    ORDER BY prix_unitaire_fcfa DESC
    LIMIT 10
""").show(truncate=False)

+---------------------------+------------------+
|nom_produit                |prix_unitaire_fcfa|
+---------------------------+------------------+
|Hisense Informatique 0593  |844000            |
|Royal Informatique 0439    |823500            |
|LG Informatique 0469       |813500            |
|Kirène Informatique 0577   |707000            |
|Adidas Informatique 0383   |698000            |
|HP Électroménager 0377     |587500            |
|Sunu Tech Informatique 0340|587000            |
|Sunu Tech Informatique 0180|586000            |
|Infinix Informatique 0445  |584000            |
|Samsung Électroménager 0063|571000            |
+---------------------------+------------------+



### B4 — Commandes livrées, canal mobile, décembre 2025
Combien de commandes `livree` du canal `mobile_app` en décembre 2025 ?

In [51]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE statut = 'livrée'
      AND canal = 'mobile_app'
      AND YEAR(date_commande) = 2025
      AND MONTH(date_commande) = 12
""").show()

+----+
|  nb|
+----+
|1745|
+----+



### B5 — Emails manquants
Combien de clients ont un email `NULL` **ou** égal à `'N/A'` ? (Rappel séance 3 : le manquant a deux visages — et `= NULL` ne fonctionne pas.)

In [52]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM clients
    WHERE email IS NULL OR email = 'N/A'
""").show()

+---+
| nb|
+---+
|150|
+---+



## Partie C — L'enquête FCFA (30 min)

### C1 — Le symptôme : la somme naïve
Calculez le CA total directement sur la colonne brute, et **notez le résultat**.

In [53]:
# === À COMPLÉTER ===
ca_naif = spark.sql("""
    SELECT SUM(montant_total_fcfa) AS ca FROM commandes
""").first()["ca"]
releves["C1_ca_naif"] = ca_naif
print(f"CA naif : {ca_naif:,.0f}")

CA naif : 11,519,493,000


### C2 — Diagnostiquer
Comptez les valeurs de `montant_total_fcfa` qui ne sont **pas** de purs nombres, puis affichez 10 valeurs fautives distinctes.

In [54]:
# === À COMPLÉTER ===
nb_pollues = spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE montant_total_fcfa NOT rlike '^[0-9]+$'
""").first()["nb"]
releves["C2_nb_valeurs_polluees"] = nb_pollues
print("valeurs polluees :", nb_pollues)

spark.sql("""
    SELECT DISTINCT montant_total_fcfa
    FROM commandes
    WHERE montant_total_fcfa NOT rlike '^[0-9]+$'
    LIMIT 10
""").show(truncate=False)

valeurs polluees : 500
+------------------+
|montant_total_fcfa|
+------------------+
|845500 FCFA       |
|554000 FCFA       |
|87700 FCFA        |
|21000 FCFA        |
|245500 FCFA       |
|154000 FCFA       |
|187000 FCFA       |
|57000 FCFA        |
|56700 FCFA        |
|1123900 FCFA      |
+------------------+



**Question C** — Expliquez en deux phrases pourquoi la requête C1 rend un résultat **faux sans lever d'erreur**.

*Votre réponse :*

…

### C3 — Nettoyer : la vue `commandes_clean`
Complétez la regex : supprimer **tout ce qui n'est pas un chiffre**, puis caster en `BIGINT`. On conserve la colonne brute sous `montant_raw`.

In [55]:
# === À COMPLÉTER ===
spark.sql("""
    CREATE OR REPLACE TEMP VIEW commandes_clean AS
    SELECT order_id, customer_id, date_commande, statut, canal,
           frais_livraison_fcfa,
           montant_total_fcfa AS montant_raw,
           CAST(regexp_replace(trim(montant_total_fcfa),
                '[^0-9]', '') AS BIGINT) AS montant_fcfa
    FROM commandes
""")
spark.sql("SELECT montant_raw, montant_fcfa FROM commandes_clean LIMIT 5").show()

+-----------+------------+
|montant_raw|montant_fcfa|
+-----------+------------+
|     190500|      190500|
|      32200|       32200|
|      40500|       40500|
|       4000|        4000|
|      35500|       35500|
+-----------+------------+



### C4 — Valider : mesurer, pas affirmer
Vérifiez qu'aucun `NULL` n'a été produit, contrôlez `MIN`/`MAX`, et recalculez le CA.

In [56]:
# === À COMPLÉTER ===
validation = spark.sql("""
    SELECT COUNT(*)                        AS nb_lignes,
           COUNT(montant_fcfa)             AS nb_castes,
           COUNT(*) - COUNT(montant_fcfa)  AS nb_null,
           MIN(montant_fcfa)               AS mini,
           MAX(montant_fcfa)               AS maxi,
           SUM(montant_fcfa)               AS ca_total
    FROM commandes_clean
""")
validation.show()
releves["C4_ca_nettoye"] = validation.first()["ca_total"]

+---------+---------+-------+----+-------+-----------+
|nb_lignes|nb_castes|nb_null|mini|   maxi|   ca_total|
+---------+---------+-------+----+-------+-----------+
|    50000|    50000|      0| 500|4182000|11645231000|
+---------+---------+-------+----+-------+-----------+



### C5 — La preuve chiffrée
Calculez l'écart entre le CA naïf (C1) et le CA nettoyé (C4), en FCFA et en pourcentage.

In [57]:
# === À COMPLÉTER ===
ecart = releves["C4_ca_nettoye"] - releves["C1_ca_naif"]
releves["C5_ecart_fcfa"] = ecart
releves["C5_ecart_pct"]  = 100 * ecart / releves["C1_ca_naif"]
print(f"Ecart : {ecart:,.0f} FCFA soit {releves['C5_ecart_pct']:.2f} %")

Ecart : 125,738,000 FCFA soit 1.09 %


## Partie D — Les indicateurs de la direction (40 min)

Toutes les requêtes portent sur `commandes_clean` (et `paiements` pour D5). Après chaque résultat, ajoutez **une phrase d'interprétation métier** dans la cellule markdown qui suit.

### D1 — CA et commandes par statut
Quelle part du CA est réellement `livree` ?

In [58]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT statut,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY statut
    ORDER BY ca_fcfa DESC
""").show()

d1 = spark.sql("""
    SELECT SUM(CASE WHEN statut = 'livree' THEN montant_fcfa ELSE 0 END) AS ca_livree,
           SUM(montant_fcfa) AS ca_total
    FROM commandes_clean
""").first()
releves["D1_part_ca_livree_pct"] = 100 * d1["ca_livree"] / d1["ca_total"]

+---------+------------+----------+
|   statut|nb_commandes|   ca_fcfa|
+---------+------------+----------+
|   livrée|       38890|9090757800|
|  annulée|        4568|1066017300|
| en_cours|        4058| 911567000|
|retournée|        2484| 576888900|
+---------+------------+----------+



*Votre réponse :*

…

### D2 — Le CA mensuel des commandes livrées
`date_trunc('month', ...)`, tri chronologique. Repérez la tendance et le mois record.

In [59]:
# === À COMPLÉTER ===
ca_mensuel = spark.sql("""
    SELECT date_trunc('month', date_commande) AS mois,
           COUNT(*)                           AS nb_commandes,
           SUM(montant_fcfa)                  AS ca_fcfa
    FROM commandes_clean
    WHERE statut = 'livrée'
    GROUP BY date_trunc('month', date_commande)
    ORDER BY mois
""")
ca_mensuel.show(24, truncate=False)

ligne_record = ca_mensuel.orderBy(ca_mensuel.ca_fcfa.desc()).first()
releves["D2_mois_record"] = str(ligne_record["mois"])

+-------------------+------------+---------+
|mois               |nb_commandes|ca_fcfa  |
+-------------------+------------+---------+
|2024-07-01 00:00:00|1230        |293487800|
|2024-08-01 00:00:00|1240        |288891300|
|2024-09-01 00:00:00|1318        |305448500|
|2024-10-01 00:00:00|1347        |321980200|
|2024-11-01 00:00:00|1319        |306182200|
|2024-12-01 00:00:00|2111        |490298700|
|2025-01-01 00:00:00|1430        |327247800|
|2025-02-01 00:00:00|1262        |288641000|
|2025-03-01 00:00:00|1457        |318877600|
|2025-04-01 00:00:00|1393        |332621300|
|2025-05-01 00:00:00|1524        |367180800|
|2025-06-01 00:00:00|1516        |353753900|
|2025-07-01 00:00:00|1636        |356315600|
|2025-08-01 00:00:00|1591        |396396800|
|2025-09-01 00:00:00|1565        |367504700|
|2025-10-01 00:00:00|1690        |406067000|
|2025-11-01 00:00:00|1639        |366408000|
|2025-12-01 00:00:00|2701        |634215100|
|2026-01-01 00:00:00|1736        |415160800|
|2026-02-0

*Votre réponse :*

…

### D3 — Panier moyen par canal
`ROUND(AVG(montant_fcfa), 0)` — mobile ou web, qui dépense le plus par commande ?

In [61]:
# === À COMPLÉTER ===
panier = spark.sql("""
    SELECT canal,
           COUNT(*)                    AS nb_commandes,
           SUM(montant_fcfa)           AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier.show()

ligne_mobile = panier.filter(panier.canal == "mobile_app").first()
releves["D3_panier_moyen_mobile"] = ligne_mobile["panier_moyen_fcfa"]

+----------+------------+----------+-----------------+
|     canal|nb_commandes|   ca_fcfa|panier_moyen_fcfa|
+----------+------------+----------+-----------------+
|mobile_app|       32519|7591773700|         233457.0|
|       web|       17481|4053457300|         231878.0|
+----------+------------+----------+-----------------+



*Votre réponse :*

…

### D4 — Top clients, avec HAVING
Top 10 des clients par CA (`GROUP BY customer_id`), en ne gardant que les clients dépassant **1 000 000 FCFA** de CA cumulé. Un client sort-il du lot ?

In [62]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT customer_id,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY customer_id
    HAVING SUM(montant_fcfa) > 1000000
    ORDER BY ca_fcfa DESC
    LIMIT 10
""").show()

+-----------+------------+---------+
|customer_id|nb_commandes|  ca_fcfa|
+-----------+------------+---------+
|    C000001|        3143|732953000|
|    C000002|         421| 95219500|
|    C000003|         323| 70031000|
|    C000005|         252| 69692400|
|    C000004|         284| 66843700|
|    C000006|         235| 53352600|
|    C000009|         162| 46914400|
|    C000008|         182| 44441800|
|    C000007|         180| 39903700|
|    C000013|         135| 39245500|
+-----------+------------+---------+



*Votre réponse :*

…

### D5 — Paiements par méthode
Nombre et pourcentage par méthode. Quelle part totale pour le **mobile money** (Orange Money + Wave + Free Money) ?

In [63]:
# === À COMPLÉTER ===
resultat_d5 = spark.sql("""
    SELECT methode,
           COUNT(*) AS nb,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS part_pct
    FROM paiements
    GROUP BY methode
    ORDER BY nb DESC
""")
resultat_d5.show()

mobile_money = ["Orange Money", "Wave", "Free Money"]
releves["D5_part_mobile_money_pct"] = (
    resultat_d5.filter(resultat_d5.methode.isin(mobile_money))
               .selectExpr("SUM(part_pct) AS total")
               .first()["total"]
)

+--------------------+-----+--------+
|             methode|   nb|part_pct|
+--------------------+-----+--------+
|        Orange Money|15646|    35.1|
|                Wave|13400|    30.1|
|Paiement à la liv...| 9783|    22.0|
|      Carte bancaire| 3506|     7.9|
|          Free Money| 2228|     5.0|
+--------------------+-----+--------+



*Votre réponse :*

…

## Partie E — SQL ou API ? `explain()` tranche (20 min)

### E1 — D3 en API DataFrame
Réécrivez le panier moyen par canal avec `groupBy().agg()`. Les chiffres doivent être **identiques**.

In [64]:
from pyspark.sql import functions as F

# === À COMPLÉTER ===
commandes_clean_df = spark.table("commandes_clean")

panier_api = (commandes_clean_df
    .groupBy("canal")
    .agg(
        F.count("*").alias("nb_commandes"),
        F.sum("montant_fcfa").alias("ca_fcfa"),
        F.round(F.avg("montant_fcfa"), 0).alias("panier_moyen_fcfa")
    )
    .orderBy(F.col("ca_fcfa").desc()))
panier_api.show()

+----------+------------+----------+-----------------+
|     canal|nb_commandes|   ca_fcfa|panier_moyen_fcfa|
+----------+------------+----------+-----------------+
|mobile_app|       32519|7591773700|         233457.0|
|       web|       17481|4053457300|         231878.0|
+----------+------------+----------+-----------------+



### E2 — B4 en API
La même requête « livrées / mobile / décembre 2025 », version `filter`.

In [65]:
# === À COMPLÉTER ===
nb_api = (spark.table("commandes")
    .filter(
        (F.col("statut") == "livree") &
        (F.col("canal") == "mobile_app") &
        (F.year("date_commande") == 2025) &
        (F.month("date_commande") == 12)
    )
    .count())
print(nb_api)

0


### E3 — Comparer les plans
Affichez le plan physique de la version SQL de D3 et de `panier_api`.

In [66]:
panier_sql = spark.sql("""
    SELECT canal, COUNT(*) AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier_sql.explain()
panier_api.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ca_fcfa#1270L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(ca_fcfa#1270L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2724]
      +- HashAggregate(keys=[canal#748], functions=[count(1), sum(montant_fcfa#1281L), avg(montant_fcfa#1281L)])
         +- Exchange hashpartitioning(canal#748, 200), ENSURE_REQUIREMENTS, [plan_id=2721]
            +- HashAggregate(keys=[canal#748], functions=[partial_count(1), partial_sum(montant_fcfa#1281L), partial_avg(montant_fcfa#1281L)])
               +- Project [canal#748, cast(regexp_replace(trim(montant_total_fcfa#750, None), [^0-9], , 1) as bigint) AS montant_fcfa#1281L]
                  +- FileScan csv [canal#748,montant_total_fcfa#750] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/amedt/Pictures/bigdata-isi-Mouhamed-THIAM/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<canal:string,m

**Question E** — Les plans physiques sont-ils identiques ? Où voit-on le filtre poussé vers la lecture (`PushedFilters` / `Filter` près du `FileScan`) ? Que concluez-vous sur le choix SQL vs API ? (3 phrases)

*Votre réponse :*

…

## Partie F — Discussion guidée (10 min, en groupes)

1. Le CA naïf de C1 était faux **en silence**. Dans une vraie entreprise, qui s'en serait aperçu, quand, et à quel coût ? Proposez **deux garde-fous techniques**.
2. `commandes_clean` est une vue temporaire : que se passe-t-il demain matin au redémarrage du notebook ? Est-ce acceptable en production ? (indice : séances 6-7)
3. La direction veut le CA par **ville du client** : quelle information manque à `commandes_clean` seule, et comment l'obtiendrez-vous en séance 5 ?

*Votre réponse :*

…

## Quiz éclair (10 min)

1. Que retourne `spark.sql(...)` : une liste, un DataFrame ou un fichier ?
2. `createOrReplaceTempView` copie-t-elle les données ? Quelle est la portée de la vue ?
3. Pourquoi `WHERE email = NULL` ne renvoie-t-il jamais rien ?
4. `SUM` sur une colonne `string` polluée : erreur ou résultat faux ? Pourquoi est-ce dangereux ?
5. `WHERE` et `HAVING` : lequel filtre les groupes, lequel filtre les lignes ?

*Notez votre score dans la cellule suivante.*

*Votre réponse :*

…

## Pour finir : relevés et livrable

In [67]:
print("=" * 60)
print("TABLEAU DE RELEVES — TP4")
print("=" * 60)
for k, v in releves.items():
    print(f"{k:32s} : {v}")

manquants = [k for k, v in releves.items() if v is None]
print("\nReleves manquants :", manquants if manquants else "aucun — bravo !")

TABLEAU DE RELEVES — TP4
A_nb_lignes_clients              : 5025
A_nb_lignes_commandes            : Ellipsis
A_type_montant_total_fcfa        : Ellipsis
C2_nb_valeurs_polluees           : 500
C1_ca_naif                       : 11519493000.0
C4_ca_nettoye                    : 11645231000
C5_ecart_fcfa                    : 125738000.0
C5_ecart_pct                     : 1.091523732858729
D1_part_ca_livree_pct            : 0.0
D2_mois_record                   : 2025-12-01 00:00:00
D3_panier_moyen_mobile           : 233457.0
D5_part_mobile_money_pct         : 70.2

Releves manquants : aucun — bravo !


### Pousser le livrable

Depuis la racine de votre dépôt :
```
git status                       # verifier que data/ n'apparait PAS
git add notebooks/TP4_spark_sql.ipynb
git commit -m "TP4 : requetes SQL et nettoyage FCFA"
git push
```

**Checklist finale**
- [ ] Notebook exécuté de bout en bout (Restart & Run All), sorties visibles ;
- [ ] `nb_null = 0` dans la validation C4 ;
- [ ] Écart CA naïf / nettoyé relevé en FCFA **et** en % ;
- [ ] Une phrase d'interprétation sous chaque indicateur de la partie D ;
- [ ] Aucun relevé manquant dans la cellule ci-dessus ;
- [ ] Données non commitées.

*Séance 5 : les jointures — lecture préalable : Damji et al., Learning Spark 2e éd., chapitre 5.*